# 🚀 Setup — Conexão ADLS Gen2 · Squad 3 (Batch)

**Projeto:** Estágio Engenharia de Dados  
**Squad:** 3 — Batch  
**Autor:** Alexandro Silva  
**Ambiente:** Databricks Serverless (Unity Catalog)  

---

## 📋 Visão Geral

Este notebook realiza a **ingestão inicial dos dados brutos** do Azure Data Lake Storage Gen2 para o Unity Catalog do Databricks.

**Fluxo:**
1. Autenticação no ADLS via Service Principal (Client Secret)
2. Listagem dos arquivos no container `raw/batch-data/`
3. Leitura dos CSVs e gravação como tabelas em `workspace.squad3.*`

> ⚠️ **Segurança:** Nunca salve credenciais diretamente no notebook.  
> Preencha os widgets manualmente a cada execução. As credenciais **não** são versionadas no GitHub.

In [0]:
# Instalaçao Bibliotecas
%pip install azure-storage-file-datalake azure-identity pymssql sqlalchemy

In [0]:
# Celula 2 — Widgets e variáveis:
from dotenv import load_dotenv
import os

dbutils.widgets.removeAll()
load_dotenv("/Workspace/Users/alexandrofs31@gmail.com/.env")

CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
TENANT_ID       = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT", "internshipdatalake")
CONTAINER       = os.getenv("ADLS_CONTAINER", "raw")
SQL_HOST        = os.getenv("SQL_HOST")
SQL_DATABASE    = os.getenv("SQL_DATABASE")
SQL_USERNAME    = os.getenv("SQL_USERNAME")
SQL_PASSWORD    = os.getenv("SQL_PASSWORD")

print("✅ Variáveis carregadas.")
print(f"   CLIENT_ID     : {'✅' if CLIENT_ID else '❌ não encontrado'}")
print(f"   TENANT_ID     : {'✅' if TENANT_ID else '❌ não encontrado'}")
print(f"   CLIENT_SECRET : {'✅' if CLIENT_SECRET else '❌ não encontrado'}")
print(f"   SQL_HOST      : {'✅' if SQL_HOST else '❌ não encontrado'}")

In [0]:
# Célula 3 - Conexão ADLS e listagem:

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential
)

filesystem_client = service_client.get_file_system_client(CONTAINER)

print(f"📂 Arquivos em '{CONTAINER}/batch-data':\n")
for item in filesystem_client.get_paths(path="batch-data"):
    tipo = "📁" if item.is_directory else "📄"
    print(f"  {tipo} {item.name}")

In [0]:
# Célula 4 — Leitura e gravação no Unity Catalog:

arquivos = {
    "ecommerce_categorias":            "batch-data/ecommerce_categorias.csv",
    "ecommerce_clientes":              "batch-data/ecommerce_clientes.csv",
    "ecommerce_enderecos":             "batch-data/ecommerce_enderecos.csv",
    "ecommerce_itens_pedido":          "batch-data/ecommerce_itens_pedido.csv",
    "ecommerce_pedidos":               "batch-data/ecommerce_pedidos.csv",
    "ecommerce_produtos":              "batch-data/ecommerce_produtos.csv",
    "ecommerce_rastreamento_entregas": "batch-data/ecommerce_rastreamento_entregas.csv",
    "food_avaliacoes_produto":         "batch-data/food_avaliacoes_produto.csv",
    "food_estoque_lojas":              "batch-data/food_estoque_lojas.csv",
    "food_fornecedores":               "batch-data/food_fornecedores.csv",
    "food_lotes_producao":             "batch-data/food_lotes_producao.csv",
    "physical_itens_venda_caixa":      "batch-data/physical_itens_venda_caixa.csv",
    "physical_lojas":                  "batch-data/physical_lojas.csv",
    "physical_produtos_pereciveis":    "batch-data/physical_produtos_pereciveis.csv",
    "physical_vendas_caixa":           "batch-data/physical_vendas_caixa.csv",
}

opcoes_adls = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.squad3")
erros = []

for nome, caminho in arquivos.items():
    try:
        url = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{caminho}"
        df = (spark.read
                .options(**opcoes_adls)
                .option("header", "true")
                .option("inferSchema", "true")
                .csv(url))
        (df.write
           .mode("overwrite")
           .option("overwriteSchema", "true")
           .saveAsTable(f"workspace.squad3.{nome}"))
        print(f"✅ workspace.squad3.{nome}  ({df.count():,} linhas)")
    except Exception as e:
        erros.append(nome)
        print(f"❌ {nome} — {str(e)[:150]}")

if erros:
    print(f"\n⚠️  Erros em {len(erros)} tabela(s): {erros}")
else:
    print("\n🎉 Todas as 15 tabelas salvas no Unity Catalog!")